# 🤖 Ejercicio: Clustering con K-Means

**K-Means** es un algoritmo de aprendizaje **no supervisado** que agrupa datos en *K* clusters, donde cada punto pertenece al cluster con el centroide más cercano.

## 📌 Objetivo del ejercicio
Segmentar clientes de una tienda retail según su **frecuencia de compra** y **monto gastado**, usando K-Means para identificar perfiles de cliente.

---

## Pasos del ejercicio
1. Generar datos sintéticos de clientes
2. Explorar y visualizar los datos
3. Determinar el número óptimo de clusters (Método del Codo)
4. Entrenar el modelo K-Means
5. Visualizar los clusters resultantes
6. Interpretar los resultados

## Paso 1 — Importar librerías

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Semilla para reproducibilidad
np.random.seed(42)
print('✅ Librerías cargadas correctamente')

ModuleNotFoundError: No module named 'pandas'

## Paso 2 — Generar datos sintéticos de clientes

Simularemos 300 clientes con tres perfiles distintos:
- 🟦 **Clientes frecuentes / alto gasto** (VIP)
- 🟩 **Clientes ocasionales / gasto medio** (Regulares)
- 🟥 **Clientes esporádicos / bajo gasto** (Inactivos)

In [ ]:
# Grupo 1: VIP — alta frecuencia, alto gasto
g1_frecuencia = np.random.normal(loc=20, scale=2, size=100)
g1_gasto      = np.random.normal(loc=5000, scale=400, size=100)

# Grupo 2: Regulares — frecuencia media, gasto medio
g2_frecuencia = np.random.normal(loc=10, scale=2, size=100)
g2_gasto      = np.random.normal(loc=2000, scale=300, size=100)

# Grupo 3: Inactivos — baja frecuencia, bajo gasto
g3_frecuencia = np.random.normal(loc=3, scale=1, size=100)
g3_gasto      = np.random.normal(loc=500, scale=150, size=100)

# Unir en un DataFrame
frecuencia = np.concatenate([g1_frecuencia, g2_frecuencia, g3_frecuencia])
gasto      = np.concatenate([g1_gasto,      g2_gasto,      g3_gasto])

df = pd.DataFrame({
    'Frecuencia_Compras': frecuencia.round(1),
    'Gasto_Total_MXN':    gasto.round(0)
})

print(f'📊 Dataset generado: {df.shape[0]} clientes, {df.shape[1]} variables')
df.head(10)

## Paso 3 — Análisis exploratorio (EDA)

In [ ]:
print('=== Estadísticas descriptivas ===')
print(df.describe().round(2))
print(f'\n¿Valores nulos? {df.isnull().sum().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot sin clusters
axes[0].scatter(df['Frecuencia_Compras'], df['Gasto_Total_MXN'],
                alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5, s=60)
axes[0].set_title('Datos sin etiquetar', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Frecuencia de Compras (veces/mes)')
axes[0].set_ylabel('Gasto Total (MXN)')
axes[0].grid(alpha=0.3)

# Histograma de gasto
axes[1].hist(df['Gasto_Total_MXN'], bins=30, color='coral', edgecolor='white', linewidth=0.5)
axes[1].set_title('Distribución del Gasto Total', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gasto Total (MXN)')
axes[1].set_ylabel('Frecuencia')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('💡 Se pueden intuir agrupaciones naturales en el scatter plot.')

## Paso 4 — Escalar los datos

K-Means usa **distancias euclidianas**, por lo que es sensible a la escala de las variables. Normalizamos con `StandardScaler`.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print('Datos originales (primeras 3 filas):')
print(df.head(3).to_string())
print('\nDatos escalados (primeras 3 filas):')
print(pd.DataFrame(X_scaled, columns=df.columns).head(3).round(3).to_string())

## Paso 5 — Método del Codo (Elbow Method)

Probamos K de 1 a 10 y graficamos la **inercia** (suma de distancias cuadradas al centroide). El "codo" indica el K óptimo.

In [ ]:
inercias   = []
siluetas   = []
rango_k    = range(2, 11)

for k in rango_k:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica del Codo
axes[0].plot(list(rango_k), inercias, 'o-', color='steelblue', linewidth=2.5, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K óptimo = 3')
axes[0].set_title('Método del Codo', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Puntuación Silueta
axes[1].plot(list(rango_k), siluetas, 's-', color='coral', linewidth=2.5, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K óptimo = 3')
axes[1].set_title('Puntuación Silueta', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Score Silueta (mayor = mejor)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

k_optimo = list(rango_k)[np.argmax(siluetas)]
print(f'✅ K óptimo sugerido por silueta: K = {k_optimo}')

## Paso 6 — Entrenar el modelo K-Means con K=3

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    init='k-means++',   # Inicialización inteligente de centroides
    n_init=10,          # 10 inicializaciones distintas
    max_iter=300,
    random_state=42
)

kmeans.fit(X_scaled)
df['Cluster'] = kmeans.labels_

print(f'Inercia final: {kmeans.inertia_:.2f}')
print(f'Iteraciones realizadas: {kmeans.n_iter_}')
print(f'\nDistribución de clientes por cluster:')
print(df['Cluster'].value_counts().sort_index())

## Paso 7 — Visualizar los clusters

In [ ]:
colores  = {0: '#E74C3C', 1: '#2ECC71', 2: '#3498DB'}
nombres  = {0: 'Grupo A', 1: 'Grupo B', 2: 'Grupo C'}

# Obtener centroides en escala original
centroides_orig = scaler.inverse_transform(kmeans.cluster_centers_)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for c in [0, 1, 2]:
    mask = df['Cluster'] == c
    axes[0].scatter(
        df.loc[mask, 'Frecuencia_Compras'],
        df.loc[mask, 'Gasto_Total_MXN'],
        color=colores[c], alpha=0.7,
        edgecolors='white', linewidth=0.5,
        s=70, label=nombres[c]
    )

# Dibujar centroides
for i, centro in enumerate(centroides_orig):
    axes[0].scatter(centro[0], centro[1],
                    color='black', marker='X', s=200, zorder=5,
                    edgecolors='white', linewidth=1.5)

axes[0].set_title('Clusters K-Means (K=3)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Frecuencia de Compras (veces/mes)')
axes[0].set_ylabel('Gasto Total (MXN)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot de gasto por cluster
data_box = [df.loc[df['Cluster']==c, 'Gasto_Total_MXN'].values for c in [0,1,2]]
bp = axes[1].boxplot(data_box, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colores.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_title('Distribución de Gasto por Cluster', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Gasto Total (MXN)')
axes[1].set_xticklabels([nombres[c] for c in [0,1,2]])
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Paso 8 — Interpretar los clusters

In [ ]:
resumen = df.groupby('Cluster')[['Frecuencia_Compras', 'Gasto_Total_MXN']].agg(['mean', 'std', 'count'])
resumen.columns = ['Freq_Media', 'Freq_StD', 'Freq_N', 'Gasto_Media', 'Gasto_StD', 'Gasto_N']
resumen['N_Clientes'] = resumen['Freq_N']
resumen = resumen.drop(columns=['Freq_N', 'Gasto_N'])
print('=== Resumen estadístico por cluster ===')
print(resumen.round(1).to_string())

# Asignar etiqueta según la media de gasto
orden = resumen['Gasto_Media'].sort_values()
etiquetas = {}
nombres_perfil = ['🔴 Inactivos (bajo gasto)', '🟡 Regulares (gasto medio)', '🟢 VIP (alto gasto)']
for i, cluster_id in enumerate(orden.index):
    etiquetas[cluster_id] = nombres_perfil[i]

print('\n=== Identificación de perfiles ===')
for k, v in sorted(etiquetas.items()):
    print(f'  Cluster {k}: {v}')

## Paso 9 — Predecir el cluster de un nuevo cliente

In [ ]:
# Nuevo cliente: compra 18 veces/mes y gasta $4,800 MXN
nuevo_cliente = np.array([[18, 4800]])
nuevo_escalado = scaler.transform(nuevo_cliente)
cluster_predicho = kmeans.predict(nuevo_escalado)[0]

print(f'Nuevo cliente → Frecuencia: 18 compras/mes | Gasto: $4,800 MXN')
print(f'Cluster asignado: {cluster_predicho}  →  {etiquetas[cluster_predicho]}')

## ✅ Conclusiones

| Concepto | Descripción |
|---|---|
| **Algoritmo** | K-Means agrupa datos minimizando la inercia (distancia al centroide) |
| **Elección de K** | El método del codo y el score silueta ayudan a elegir K |
| **Escalado** | Obligatorio cuando las variables tienen escalas distintas |
| **k-means++** | Inicialización que acelera la convergencia y da mejores resultados |
| **Limitación** | K-Means asume clusters esféricos; no funciona bien con formas irregulares |

---

## 🧩 Retos adicionales
1. Prueba con `K=4` o `K=5` — ¿los perfiles son más útiles?
2. Agrega una tercera variable (p. ej. `Dias_Ultimo_Compra`) y repite el clustering
3. Compara K-Means con **DBSCAN** o **AgglomerativeClustering** sobre los mismos datos
4. Exporta el DataFrame final a CSV con `df.to_csv('clientes_segmentados.csv', index=False)`